## dfl_train.py  --  Decision-Focused Learning training loop.

Fine-tunes the prosumption forecaster end-to-end against downstream battery-dispatch
regret. Built against the CURRENT validated state:
  - nominal-LDR (A) dispatch layer (no robust box)
  - gamma = 1e-4  (conditions k=1; economic decisions pinned to ~1e-6)
  - ECOS for the differentiable layer forward (diffcp: interior-point, accurate gradients)
  - clip_recourse=True in the loss  (ex-post feasibility is the training signal)

Loss vs metric:
  TRAIN on realised_cost (differentiable). REPORT regret = realised_cost - oracle_cost
  (oracle detached, so grad(regret) == grad(realised_cost)). Early-stop on VAL regret.

Strategy: get ONE config working end-to-end (dual, k=0) -> train, val-eval, sane regret,
THEN replicate to the four corners from the SAME baseline warm-start. Do NOT cold-start four.

cvxpy/torch/ecos run in the user's env; this file is parse/logic-validated, not executed.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import copy
import numpy as np
import pandas as pd
import torch
import cvxpy as cp

import sys
import pickle
from pathlib import Path
from pyprojroot import here

ROOT_DIR = here()
FORECASTING_DIR = ROOT_DIR / "4_forecasting"
DATA_DIR = ROOT_DIR / "1_data" / "processed"
COPULA_DIR = ROOT_DIR / "5_scenario_gen"
MODEL_DIR = ROOT_DIR / "6_models"
DFL_TRAIN_DIR = ROOT_DIR / "7_model_training"

sys.path.insert(0, str(FORECASTING_DIR))  # make forecasting module importable
sys.path.insert(0, str(COPULA_DIR))  # make copula module importable
sys.path.insert(0, str(MODEL_DIR))  # make model modules importable
sys.path.insert(0, str(DFL_TRAIN_DIR)) # make training script importable

from forecasting import (reindex_and_impute, build_features, make_windows,
                            normalise_hist, normalise_exo, denormalise_y, normalise_y,
                            pinball_loss, QUANTILE_LEVELS, Baseline_Forecaster,
                            TRAIN_START, VAL_START, TEST_START,
                            HIST_COLS, FEAT_COLS, EXO_COLS)
from copula_lib import FrozenCopulaSampler

from dispatch_layer import (default_fixed_params, build_problem, make_layer,
                            build_oracle, solve_oracle)
from dispatch_wrapper import (get_prices, oracle_price_values, realised_cost, realised_breakdown,
                            cholesky_of_second_moment)
from diffcp import SolverError

In [ ]:
PRICE_COLS = ["da", "imb", "up_reg_cost", "down_reg_cost"]
ISSUE_HOUR, HORIZON, N_HIST = 9, 24, 168        # N_HIST -> your forecaster's lookback

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_SCEN       = 64
GAMMA        = 1e-4
TRAIN_SOLVER = "ECOS"          
ORACLE_SOLVER = cp.GUROBI

QUANTILE_LEVELS_TENSOR = torch.as_tensor(QUANTILE_LEVELS, dtype=torch.float32, device=device)
EPS_BALANCE = 1e-6          # guards self_balanced_loss's denominator against 0/0
FALLBACK_SOLVER = "SCS"     # tried if TRAIN_SOLVER ("ECOS") errors


@dataclass
class TrainConfig:
    price_model: str           # "single" | "dual"
    k: float                   # 0.0 | 1.0
    lr: float = 5e-4
    batch_size: int = 16       # days per optimiser step (gradient accumulation)
    max_epochs: int = 50       # bounded cap (baseline forecaster used 200/uncapped;
                                # kept bounded here since k=1's imbalance loss is new)
    patience: int = 10         # epochs of no val_fsurr improvement before stop
                                # (aligned with baseline forecaster's early_stopping.patience)
    min_delta: float = 0.02    # RELATIVE improvement required to reset patience (fraction of
                                # best_val, e.g. 0.02 = 2%) -- NOT absolute. Early stopping and
                                # checkpoint selection are keyed on val_fsurr (the decision-loss
                                # surrogate: regret for k=0, imbalance for k=1) -- the metric
                                # DFL is actually trying to improve -- rather than the
                                # self-balanced val_combined. val_combined's weights are
                                # UNFLOORED (self_balanced_loss), so as one term grows its own
                                # weight collapses toward 0, making val_combined numerically
                                # insensitive to exactly the kind of blow-up seen in the
                                # single/k=0 verification run (val_pinball 147->760 over 15
                                # epochs while val_combined barely moved). val_fsurr doesn't
                                # have that problem. 2% is grounded in the pre-floor single/k=0
                                # run's ~0.3-1.4% epoch-to-epoch noise band.
    grad_clip: float = 5.0     # max grad norm (DFL stability); None to disable
    seed: int = 20240801

In [ ]:
# =====================================================================================
# Forward: forecaster -> quantiles (GRAD KEPT) -> mean/xi -> layer inputs.
# NOTE the difference from the frozen notebook forecast_fn: NO torch.no_grad() here, so
# the gradient flows quantiles -> forecaster weights. Output cast to float64 to match the
# validated layer path (grad flows through the cast to the float32 weights).
# =====================================================================================
def forecast_train(model, sc, x_hist_day, x_fut_day, device, normalise_hist, denormalise_y,
                    normalise_exo):
    xh = normalise_hist(np.asarray(x_hist_day), sc)
    xh = torch.as_tensor(xh, dtype=torch.float32, device=device).unsqueeze(0)
    xf = normalise_exo(np.asarray(x_fut_day), sc)
    xf = torch.as_tensor(xf, dtype=torch.float32, device=device).unsqueeze(0)
    q_norm = model(xh, xf)                                  # (1, K, Q), grad -- STANDARDISED
    q_phys = denormalise_y(q_norm, sc)
    # Both from the SAME forward pass (dropout is active in train() mode -- a second call
    # would sample a different mask and decouple the anchor from the actual decision).
    return q_phys.squeeze(0).to(torch.float64), q_norm.squeeze(0)   # (K,Q) f64, (K,Q) f32

def build_layer_vals(mean, xi, prices, fp, device):
    """Named param values (torch) for the layer forward. pl_hat/xi/Sigma_xi_chol carry
    grad; prices do not. solve/forward select the subset actually needed by name (extra
    unused keys here are harmless). xi_samples is only consumed by dual+k<1's economic
    epigraph; Sigma_xi_chol (differentiable Cholesky of xi^T xi/N, see
    dispatch_wrapper.cholesky_of_second_moment) is what k>0's tracking term consumes --
    ~24x faster to solve than the old xi_samples-based tracking term, identical value by
    the trace identity sum_squares(R@xi^T)/N == sum_squares(R@Chol). See
    dispatch_layer.build_problem."""
    vals = {"pl_hat": mean, "xi_samples": xi}              # grad-carrying
    if fp.k > 0.0:
        vals["Sigma_xi_chol"] = cholesky_of_second_moment(xi)   # grad-carrying

    # Now add in all the price data
    for kk, vv in prices.items():
        vals[kk] = torch.as_tensor(np.asarray(vv, float), dtype=mean.dtype, device=device)
    return vals

def imbalance_loss(fp, bd):
    """L2 realised-imbalance loss for k=1 training: sum_t p_imb[t]**2 * dt.
    Mirrors the k=1 dispatch objective's own sum_squares tracking term, so the training
    loss and the decisions it supervises are the same penalty family. dim=-1 reduces
    correctly for both unbatched p_imb (T,) -> scalar and batched (B,T) -> (B,), exactly
    like C_da/C_imb already reduce inside realised_breakdown."""
    return (bd.p_imb ** 2).sum(dim=-1) * fp.dt

def pinball_per_day(y_true_norm, q_norm, levels):
    """Per-day pinball loss (B,) or scalar, mirroring forecasting.pinball_loss's formula
    WITHOUT its final batch-mean -- self-balancing needs per-day granularity, not a
    batch-averaged scalar. y_true_norm: (B,K), q_norm: (B,K,Q), levels: (Q,) tensor."""
    e = y_true_norm.unsqueeze(-1) - q_norm
    q = levels.view(1, 1, -1)
    return torch.maximum(q * e, (q - 1.0) * e).sum(dim=(1, 2)).to(torch.float64)

def self_balanced_loss(L_base, f_dfl, eps=EPS_BALANCE):
    """alpha*L_base + beta*f_dfl, weights detached and UNFLOORED: alpha=f_dfl/denom,
    beta=L_base/denom, denom=L_base+f_dfl+eps -- so alpha*L_base == beta*f_dfl exactly
    (each term is always exactly half the loss VALUE). This is the paper's formulation as
    specified; NOTE it says nothing about gradient magnitude, and if one term sits near a
    structural floor (observed for single/k=0's f_dfl=regret, pinned near-optimal by
    gamma=1e-4 almost independent of forecast quality) its weight collapses toward 0 as the
    other term grows. That's why early stopping/checkpoint selection (train_one_config) is
    keyed on val_fsurr directly, not on this combined value -- this combined loss is used
    ONLY for the training gradient, per the paper."""
    denom = L_base.detach() + f_dfl.detach() + eps
    alpha = f_dfl.detach() / denom
    beta  = 1.0 - alpha
    return alpha * L_base + beta * f_dfl

def solve_with_retry(layer, args):
    """Try TRAIN_SOLVER (ECOS), then FALLBACK_SOLVER (SCS). Raises the SCS SolverError if
    both fail -- caller decides whether to skip."""
    try:
        return layer(*args, solver_args={"solve_method": TRAIN_SOLVER})
    except SolverError:
        return layer(*args, solver_args={"solve_method": FALLBACK_SOLVER})

def dfl_loss_one_day(d, *, model, sampler, sc, windows, bundle, layer, keys, fp,
                     price_model, device, fwd, oracle_cost=None, return_components=False):
    """Differentiable self-balanced loss for one day: alpha*L_base + beta*f_dfl, where
    L_base is the pinball-loss calibration anchor and f_dfl is regret (realised_cost -
    oracle_cost) for k=0 or raw imbalance_loss for k=1 (already >= 0, already "regret vs
    the trivial free-bid imbalance oracle", which is analytically zero -- see Phase 3 notes).
    `fwd` bundles the forecasting helpers."""
    realised  = np.asarray(windows.y[d], float)
    price_day = np.asarray(windows.price[d], float)
    q_phys, q_norm = forecast_train(model, sc, windows.x_hist[d], windows.x_fut[d], device,
                               fwd["normalise_hist"], fwd["denormalise_y"], fwd["normalise_exo"])
    mean, xi = sampler.mean_and_errors(q_phys)          # grad in quantiles (physical scale)
    prices = get_prices(price_day, price_model)
    vals = build_layer_vals(mean, xi, prices, fp, device)
    args = [vals[name] for name in keys]                   # bundle-order params
    dec = solve_with_retry(layer, args)                     # raises if both ECOS & SCS fail

    y_true_norm = torch.as_tensor(normalise_y(realised, sc), dtype=torch.float32,
                                   device=device).unsqueeze(0)
    L_base = pinball_per_day(y_true_norm, q_norm.unsqueeze(0), QUANTILE_LEVELS_TENSOR).squeeze(0)

    if fp.k == 0.0:
        assert oracle_cost is not None, "oracle_cost required for k=0 (need regret, not raw cost)"
        raw_cost = realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                             realised=realised, pl_hat=mean, price_model=price_model,
                             clip_recourse=True, **prices)
        # clamp against tiny numerical noise (ECOS-vs-GUROBI precision differences can push
        # near-zero regret marginally negative; the theoretical lower bound still holds)
        f_dfl = torch.clamp(raw_cost - float(oracle_cost), min=0.0)
    else:  # k == 1: train on imbalance, not economic cost
        bd = realised_breakdown(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                                realised=realised, pl_hat=mean, price_model=price_model,
                                clip_recourse=True, **prices)
        f_dfl = imbalance_loss(fp, bd)                     # already >= 0

    combined = self_balanced_loss(L_base, f_dfl)
    if return_components:
        return combined, L_base, f_dfl
    return combined

def dfl_loss_batch(batch_indices, *, model, sampler, sc, windows, bundle, layer, keys, fp,
                   price_model, device, fwd, oracle_costs=None):
    """Differentiable self-balanced loss for a BATCH of days. Vectorized happy path (one
    batched realised_cost/realised_breakdown/pinball_per_day call across all B days, same
    cost profile as before Phase 3); per-day fallback (with skip) only triggers if the
    batched solve fails with BOTH ECOS and SCS. Returns (loss_sum, n_survived) -- the
    caller must divide by n_survived, not the original batch size, since skipped days
    shrink the effective batch."""
    realised  = np.asarray(windows.y[batch_indices], float)
    price_day = np.asarray(windows.price[batch_indices], float)
    x_hist = windows.x_hist[batch_indices]
    x_fut = windows.x_fut[batch_indices]

    B = len(batch_indices)
    means_list = []
    xis_list = []
    q_norm_list = []

    # 2. Sequential loop over forecaster to satisfy 2D downstream dependencies
    for i in range(B):
        # 1. Neural Network yields quantiles: [Q, T]
        q_phys_i, q_norm_i = forecast_train(model, sc, x_hist[i], x_fut[i], device,
                                     fwd["normalise_hist"], fwd["denormalise_y"], fwd["normalise_exo"])
        # Call the unmodified 2D function
        mean_i, xi_i = sampler.mean_and_errors(q_phys_i)

        means_list.append(mean_i)
        xis_list.append(xi_i)
        q_norm_list.append(q_norm_i)

    # 3. Stack into batched tensors: [B, T] and [B, N, T]
    mean = torch.stack(means_list, dim=0)
    xi = torch.stack(xis_list, dim=0)
    q_norm_batch = torch.stack(q_norm_list, dim=0)          # (B, K, Q), grad -- STANDARDISED

    # 4. Build Layer Arguments
    prices = get_prices(price_day, price_model)
    vals = build_layer_vals(mean, xi, prices, fp, device)
    args = [vals[name] for name in keys]
    y_true_norm = torch.as_tensor(normalise_y(realised, sc), dtype=torch.float32, device=device)  # (B,K)

    def _combine(dec, realised_s, mean_s, prices_s, q_norm_s, y_true_s, oracle_s):
        """VECTORIZED self-balanced loss -- realised_cost/realised_breakdown/pinball_per_day
        each called ONCE across however many days are in this call (B, or 1 in the
        fallback), exactly like the pre-Phase-3 code did. Returns a (b,) per-day tensor."""
        L_base = pinball_per_day(y_true_s, q_norm_s, QUANTILE_LEVELS_TENSOR)        # (b,)
        if fp.k == 0.0:
            raw_cost = realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                                realised=realised_s, pl_hat=mean_s, price_model=price_model,
                                clip_recourse=True, **prices_s)                      # (b,)
            # explicit tensor conversion -- subtracting a bare numpy array from a torch
            # tensor isn't reliably safe across torch versions, unlike a python float scalar
            oracle_t = torch.as_tensor(oracle_s, dtype=raw_cost.dtype, device=raw_cost.device)
            # clamp against tiny numerical noise (ECOS-vs-GUROBI precision differences can
            # push near-zero regret marginally negative; the theoretical lower bound holds)
            f_dfl = torch.clamp(raw_cost - oracle_t, min=0.0)
        else:
            bd = realised_breakdown(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                                realised=realised_s, pl_hat=mean_s, price_model=price_model,
                                clip_recourse=True, **prices_s)
            f_dfl = imbalance_loss(fp, bd)                                          # (b,)
        return self_balanced_loss(L_base, f_dfl)                                    # (b,)

    # 5. Execute Parallel Solve (batched fast path; per-day fallback only on double failure)
    try:
        dec = solve_with_retry(layer, args)                 # batched: ECOS then SCS
        oracle_arr = (np.asarray([oracle_costs[d] for d in batch_indices], float)
                      if fp.k == 0.0 else None)
        per_day = _combine(dec, realised, mean, prices, q_norm_batch, y_true_norm, oracle_arr)
        return per_day.sum(), B
    except SolverError:
        pass  # both batched attempts failed -- fall back to per-day, skipping failures only here

    total = 0.0; n_survived = 0
    for i in range(B):
        args_i = [v[i:i+1] for v in args]
        try:
            dec_i = solve_with_retry(layer, args_i)
        except SolverError:
            print(f"  SKIPPING day {batch_indices[i]} ({windows.delivery_start[batch_indices[i]]}): "
                  f"both ECOS and SCS failed")
            continue
        oracle_i = (np.asarray([oracle_costs[batch_indices[i]]], float) if fp.k == 0.0 else None)
        prices_i = {kk: v[i:i+1] for kk, v in prices.items()}
        loss_i = _combine(dec_i, realised[i:i+1], mean[i:i+1], prices_i,
                          q_norm_batch[i:i+1], y_true_norm[i:i+1], oracle_i)
        total = total + loss_i.sum()
        n_survived += 1
    if n_survived == 0:
        raise RuntimeError(f"all {B} days in this batch failed to solve with both ECOS and SCS")
    return total, n_survived

In [ ]:
# =====================================================================================
# Economic oracle costs (detached constants). k-independent and forecaster-independent,
# so compute ONCE per (day, price_model) and reuse across all corners/epochs.
# =====================================================================================
def precompute_oracle_costs(windows, price_model, phys_fp):
    ob = build_oracle(phys_fp, price_model, objective="economic")
    costs = np.empty(len(windows.delivery_start))
    for d in range(len(windows.delivery_start)):
        realised  = np.asarray(windows.y[d], float)
        price_day = np.asarray(windows.price[d], float)
        v = oracle_price_values(price_day, price_model, realised)
        costs[d] = solve_oracle(ob, v, solver=ORACLE_SOLVER)
    return costs


def evaluate_regret(*, model, sampler, sc, windows, oracle_costs, bundle, layer, keys, fp,
                    price_model, device, fwd):
    """Mean per-day self-balanced loss, decision-loss surrogate (f_dfl, already genuine
    regret for k=0 / already-nonnegative imbalance for k=1), and calibration anchor
    (L_base) over a window set (no grad). val_fsurr (decision loss) is the early-stopping
    signal used by train_one_config -- val_combined is diagnostic only, see
    TrainConfig.min_delta docstring for why. Skips (and counts) any day where both ECOS
    and SCS fail, same as training."""
    model.eval()
    tot_combined = 0.0; tot_fsurr = 0.0; tot_base = 0.0; n_ok = 0; n_skipped = 0
    with torch.no_grad():
        for d in range(len(windows.delivery_start)):
            oc = float(oracle_costs[d]) if fp.k == 0.0 else None
            try:
                combined, L_base, f_dfl = dfl_loss_one_day(d, model=model, sampler=sampler,
                                        sc=sc, windows=windows, bundle=bundle, layer=layer,
                                        keys=keys, fp=fp, price_model=price_model,
                                        device=device, fwd=fwd, oracle_cost=oc,
                                        return_components=True)
            except SolverError:
                n_skipped += 1
                print(f"  evaluate_regret: skipping val day {d} ({windows.delivery_start[d]}) "
                      f"-- both ECOS and SCS failed")
                continue
            tot_combined += float(combined); tot_fsurr += float(f_dfl); tot_base += float(L_base)
            n_ok += 1
    return tot_combined / n_ok, tot_fsurr / n_ok, tot_base / n_ok, n_skipped

In [ ]:
# =====================================================================================
# Train ONE config: warm-started model, early-stop on val DECISION loss (val_fsurr --
# regret for k=0, imbalance for k=1), restore best. NOTE: early stopping/checkpoint
# selection is keyed on val_fsurr, NOT the self-balanced val_combined -- see
# TrainConfig.min_delta docstring for why (val_combined's weights self-collapse and can
# go numerically blind to calibration blowing up; val_fsurr is also the metric DFL is
# actually trying to improve).
# =====================================================================================
def train_one_config(cfg: TrainConfig, *, model, sampler, sc, train_windows, val_windows,
                     oracle_costs_train, oracle_costs_val, device, fwd, batch_input = True):
    torch.manual_seed(cfg.seed)
    fp = default_fixed_params(cfg.k, num_scenarios=N_SCEN, gamma=GAMMA)
    bundle = build_problem(fp, cfg.price_model)
    layer  = make_layer(bundle)
    keys   = [p.name() for p in bundle.params]
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    n_train = len(train_windows.delivery_start)
    best_val = float("inf"); best_state = copy.deepcopy(model.state_dict()); patience = 0
    history = []

    def _loss_per_day(day, oracle_cost):
        return dfl_loss_one_day(day, model=model, sampler=sampler, sc=sc, windows=train_windows,
                                bundle=bundle, layer=layer, keys=keys, fp=fp,
                                price_model=cfg.price_model, device=device, fwd=fwd,
                                oracle_cost=oracle_cost)

    def _loss_per_batch(batch_list):
         return dfl_loss_batch(batch_list, model=model, sampler=sampler, sc=sc, windows=train_windows,
                                        bundle=bundle, layer=layer, keys=keys, fp=fp,
                                        price_model=cfg.price_model, device=device, fwd=fwd,
                                        oracle_costs=oracle_costs_train)
        

    for epoch in range(cfg.max_epochs):
        model.train()
        order = np.random.permutation(n_train)
        for start in range(0, n_train, cfg.batch_size):
            batch = order[start:start + cfg.batch_size]
            opt.zero_grad()
            if batch_input:
                try:
                    batch_loss, n_survived = _loss_per_batch(batch.tolist())
                except RuntimeError as e:      # all days in batch failed -- skip the whole step
                    print(f"  SKIPPING whole batch (start={start}): {e}")
                    continue
            else:
                batch_loss = 0.0; n_survived = len(batch)
                for d in batch:
                    oc = float(oracle_costs_train[d]) if cfg.k == 0.0 else None
                    batch_loss = batch_loss + _loss_per_day(int(d), oc)
            (batch_loss / n_survived).backward()
            if cfg.grad_clip is not None:               
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()

        val_combined, val_fsurr, val_base, n_skipped = evaluate_regret(
            model=model, sampler=sampler, sc=sc, windows=val_windows,
            oracle_costs=oracle_costs_val, bundle=bundle, layer=layer, keys=keys, fp=fp,
            price_model=cfg.price_model, device=device, fwd=fwd)
        history.append({"val_combined": val_combined, "val_fsurr": val_fsurr,
                        "val_base": val_base, "n_skipped": n_skipped})
        # RELATIVE improvement check on val_fsurr (decision loss -- what DFL is actually
        # trying to improve), NOT val_combined -- see TrainConfig.min_delta docstring.
        # best_val starts at inf, so the first epoch always "improves".
        improved = val_fsurr < best_val * (1.0 - cfg.min_delta)
        print(f"[{cfg.price_model} k={int(cfg.k)}] epoch {epoch:2d}  "
              f"val_fsurr={val_fsurr:.4f}  val_pinball={val_base:.4f}  "
              f"val_combined={val_combined:.4f}  skipped={n_skipped}  "
              f"{'*best' if improved else f'(patience {patience+1}/{cfg.patience})'}")
        if improved:
            best_val = val_fsurr; best_state = copy.deepcopy(model.state_dict()); patience = 0
        else:
            patience += 1
            if patience >= cfg.patience:
                print(f"  early stop at epoch {epoch} (best val_fsurr={best_val:.4f})")
                break
    

    model.load_state_dict(best_state)                       # restore best
    return model, best_val, history

# Run the training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- frame + splits (train 2018 / val 2019-H1; TEST stays SEALED) ---
base  = pd.read_csv(DATA_DIR / "df_full.csv", parse_dates=["datetime"]).set_index("datetime")
base  = reindex_and_impute(base, HIST_COLS, freq="1h", warn_gap=6)
frame = build_features(base, feature_cols=FEAT_COLS)
win_kw = dict(gate_aligned_only=True, issue_hour=ISSUE_HOUR, hist_cols=HIST_COLS,
                exo_cols=EXO_COLS, target_col="prosumption", price_cols=PRICE_COLS,
                n_hist=N_HIST, horizon=HORIZON)
train_windows = make_windows(frame, y_range=(TRAIN_START, VAL_START - pd.Timedelta(hours=1)), **win_kw)
val_windows   = make_windows(frame, y_range=(VAL_START,   TEST_START - pd.Timedelta(hours=1)), **win_kw)

# --- warm-start forecaster (fresh copy per config) + sampler ---
ckpt  = torch.load(FORECASTING_DIR / "baseline_forecaster_best.pt", weights_only=False, map_location="cpu")
def fresh_baseline():
    m = Baseline_Forecaster(**ckpt["model_config"]); m.load_state_dict(ckpt["state_dict"])
    return m.to(device)
sc = ckpt["scaler_stats"]
cop = pickle.load(open(COPULA_DIR / "frozen_copula.pkl", "rb"))
sampler = FrozenCopulaSampler(cop["Z_corr"], cop["quantile_levels"]).to(device)

fwd = {"normalise_hist": normalise_hist, "denormalise_y": denormalise_y, "normalise_exo": normalise_exo}
phys_fp = default_fixed_params(0.0, num_scenarios=N_SCEN, gamma=GAMMA)  # physical params only

# --- all four corners, each from the SAME baseline warm-start (do NOT chain between corners) ---
results = {}
for pm, k in [("single", 0.0), ("single", 1.0), ("dual", 0.0), ("dual", 1.0)]:
    cfg = TrainConfig(price_model=pm, k=k)
    print(f"\n=== training corner: price_model={cfg.price_model}  k={int(cfg.k)} ===")

    if cfg.k == 0.0:
        # k=0's self-balanced loss needs regret (realised_cost - oracle_cost), which is
        # non-negative, unlike raw realised_cost which is frequently negative in this
        # dataset -- required on BOTH train and val days now (train: inside the loss
        # itself; val: evaluate_regret's reported metrics).
        oracle_train = precompute_oracle_costs(train_windows, cfg.price_model, phys_fp)
        oracle_val   = precompute_oracle_costs(val_windows,   cfg.price_model, phys_fp)
    else:
        # k=1: no oracle needed -- imbalance_loss is already >= 0, and already equals
        # "regret vs the imbalance oracle" since that oracle is analytically zero
        # (free/clairvoyant bid, established this session).
        oracle_train = None
        oracle_val   = None

    model = fresh_baseline()
    model, best_val, hist = train_one_config(
        cfg, model=model, sampler=sampler, sc=sc,
        train_windows=train_windows, val_windows=val_windows,
        oracle_costs_train=oracle_train, oracle_costs_val=oracle_val,
        device=device, fwd=fwd)

    print(f"\nDONE {cfg.price_model}/k={int(cfg.k)}. best val_fsurr = {best_val:.4f}")
    torch.save({"state_dict": model.state_dict(), "cfg": cfg.__dict__,
                "best_val_fsurr": best_val, "val_history": hist},
                DFL_TRAIN_DIR / f"dfl_{cfg.price_model}_k{int(cfg.k)}.pt")

    results[(cfg.price_model, cfg.k)] = {"best_val": best_val, "history": hist}

print("\n=== ALL FOUR CORNERS DONE ===")
for (pm, k), r in results.items():
    print(f"  {pm:6s} k={int(k)}: best val_fsurr = {r['best_val']:.4f}  (epochs run: {len(r['history'])})")